# DS&A code walkthrough: every function in `dsa/`
**Course:** DBX Study Guide | **Duration:** ~3 hours, or one part at a time | **Runs on:** any compute (plain Python, no Spark)

This notebook goes through the `dsa/` folder file by file. Each section gives you the **problem**, the **idea** and the **complexity**, then a code cell with the
function's real source and a small example, then **what to notice** in the output.

| Part | File | What's in it |
|---|---|---|
| 1 | `arrays.py` | two pointers, Kadane, prefix/suffix, intervals, in-place tricks, sliding windows |
| 2 | `strings.py` | two pointers, counting, sliding windows, expand around center, stacks |
| 3 | `binary_search.py` | classic search, boundaries, rotated arrays, binary search on the answer |
| 4 | `graphs.py` | BFS, DFS, grids, cycles, topological sort, Dijkstra, union-find |
| 5 | `hash_maps.py` | complements, grouping, counting, prefix sums, LRU cache, hash partitioning |
| 6 | `python_optimizations.py` | 7 slow vs fast pairs |
| 7 | `benchmark.py` | measuring Big-O from real timings |

`dsa/__init__.py` only marks the folder as a Python package, so it has nothing to walk through.

> **Spoiler warning:** these are the solutions to `exercises/`. Try a problem in `01_dsa_practice` first if you want to test yourself.

### Setup
Find the repo root and put it on the Python path. Only Part 7 imports from it, because every other cell defines its function right here.

In [ ]:
import os
import sys

path = os.path.abspath(os.getcwd())
while not os.path.isdir(os.path.join(path, "dsa")):
    parent = os.path.dirname(path)
    if parent == path:
        raise RuntimeError("Repo root not found. Keep the folder structure from GitHub intact.")
    path = parent
if path not in sys.path:
    sys.path.insert(0, path)
print("Repo root:", path)

**Setup complete.**

---

# Part 1: `dsa/arrays.py`
Array questions usually ask you to replace a nested loop with **one pass** by keeping the right state. Start by loading what this file imports.

In [ ]:
from collections import deque
from typing import List, Tuple

The imports are loaded.

---

## `two_sum_sorted`
- **Problem:** In a **sorted** list, find the indices of two numbers that add up to `target`.
- **Idea:** Start a pointer at each end. If the sum is too small, only moving `lo` right can raise it. If it's too big, only moving `hi` left can lower it.
- **Complexity:** O(n) time, O(1) space

In [ ]:
def two_sum_sorted(nums: List[int], target: int) -> Tuple[int, int]:
    lo, hi = 0, len(nums) - 1
    while lo < hi:
        s = nums[lo] + nums[hi]
        if s == target:
            return lo, hi
        if s < target:
            lo += 1
        else:
            hi -= 1
    return -1, -1


# Try it
print(two_sum_sorted([1, 2, 4, 7, 11], 15))
print(two_sum_sorted([1, 2, 3], 100))

**What to notice:** The pointers tried 1+11=12, 2+11=13, then 4+11=15, so the answer is `(2, 4)`. Sorting is what makes each move safe. On unsorted input, use the hash map version in Part 5.

---

## `max_subarray`
- **Problem:** Find the largest sum of any contiguous, non-empty subarray.
- **Idea:** This is **Kadane's algorithm**. The best subarray ending at each position either extends the previous one or starts fresh, whichever is larger.
- **Complexity:** O(n) time, O(1) space

In [ ]:
def max_subarray(nums: List[int]) -> int:
    best = cur = nums[0]
    for x in nums[1:]:
        cur = max(x, cur + x)
        best = max(best, cur)
    return best


# Try it
print(max_subarray([-2, 1, -3, 4, -1, 2, 1, -5, 4]))
print(max_subarray([-3, -1, -2]))

**What to notice:** The first answer is 6, from `[4, -1, 2, 1]`. The all-negative list returns -1, not 0, because the subarray can't be empty. That's why the code starts from `nums[0]` rather than 0.

---

## `product_except_self`
- **Problem:** Return a list where each position holds the product of every **other** element, without using division.
- **Idea:** Each answer is (product of everything to the left) x (product of everything to the right). One forward pass fills in the left products, and one backward pass multiplies in the right products.
- **Complexity:** O(n) time, O(1) extra space

In [ ]:
def product_except_self(nums: List[int]) -> List[int]:
    n = len(nums)
    out = [1] * n
    prefix = 1
    for i in range(n):
        out[i] = prefix
        prefix *= nums[i]
    suffix = 1
    for i in range(n - 1, -1, -1):
        out[i] *= suffix
        suffix *= nums[i]
    return out


# Try it
print(product_except_self([1, 2, 3, 4]))
print(product_except_self([-1, 1, 0, -3, 3]))

**What to notice:** After the forward pass, the first list is `[1, 1, 2, 6]`, and the backward pass turns it into `[24, 12, 8, 6]`. Zeros need no special handling, which is a big advantage over dividing the total product.

---

## `merge_intervals`
- **Problem:** Merge all overlapping `[start, end]` intervals.
- **Idea:** Sort by start, then sweep. Each interval either overlaps the last merged one (extend its end) or starts a new one.
- **Complexity:** O(n log n) time, from the sort

In [ ]:
def merge_intervals(intervals: List[List[int]]) -> List[List[int]]:
    if not intervals:
        return []
    intervals = sorted(intervals, key=lambda iv: iv[0])
    merged = [list(intervals[0])]
    for start, end in intervals[1:]:
        if start <= merged[-1][1]:
            merged[-1][1] = max(merged[-1][1], end)
        else:
            merged.append([start, end])
    return merged


# Try it
print(merge_intervals([[1, 3], [8, 10], [2, 6], [15, 18]]))
print(merge_intervals([[1, 4], [4, 5]]))

**What to notice:** `[1, 3]` and `[2, 6]` become `[1, 6]`. Touching intervals merge too, because the check is `<=`. The same logic checks that SCD2 validity windows don't overlap.

---

## `rotate_right`
- **Problem:** Rotate a list right by `k` steps **in place**.
- **Idea:** Use three reversals: the whole list, then the first `k` elements, then the rest. `k %= n` handles `k` larger than the list.
- **Complexity:** O(n) time, O(1) space

In [ ]:
def rotate_right(nums: List[int], k: int) -> None:
    n = len(nums)
    if n == 0:
        return
    k %= n

    def rev(i: int, j: int) -> None:
        while i < j:
            nums[i], nums[j] = nums[j], nums[i]
            i, j = i + 1, j - 1

    rev(0, n - 1)
    rev(0, k - 1)
    rev(k, n - 1)


# Try it
nums = [1, 2, 3, 4, 5, 6, 7]
rotate_right(nums, 3)
print(nums)

**What to notice:** The list goes `7654321`, then `5674321`, then `5671234`. The function returns nothing because it changes the list you passed in, so print the list, not the return value.

---

## `move_zeroes`
- **Problem:** Move all zeros to the end **in place**, keeping the other values in order.
- **Idea:** `read` visits every element, and `write` marks where the next non-zero belongs. Swapping non-zeros to `write` pushes zeros back.
- **Complexity:** O(n) time, O(1) space

In [ ]:
def move_zeroes(nums: List[int]) -> None:
    write = 0
    for read in range(len(nums)):
        if nums[read] != 0:
            nums[write], nums[read] = nums[read], nums[write]
            write += 1


# Try it
nums = [0, 1, 0, 3, 12]
move_zeroes(nums)
print(nums)

**What to notice:** The non-zeros keep their order: `[1, 3, 12, 0, 0]`. This read/write pointer pattern also solves in-place dedupe and filtering.

---

## `max_sliding_window`
- **Problem:** Return the maximum of every window of size `k`.
- **Idea:** Keep a **monotonic deque** of indices with decreasing values. A new value removes smaller ones from the back, because they can never be the max again. The front is always the current max.
- **Complexity:** O(n) time, because each index is added and removed at most once; O(k) space

In [ ]:
def max_sliding_window(nums: List[int], k: int) -> List[int]:
    dq: deque = deque()
    out = []
    for i, x in enumerate(nums):
        while dq and nums[dq[-1]] <= x:
            dq.pop()
        dq.append(i)
        if dq[0] <= i - k:
            dq.popleft()
        if i >= k - 1:
            out.append(nums[dq[0]])
    return out


# Try it
print(max_sliding_window([1, 3, -1, -3, 5, 3, 6, 7], 3))

**What to notice:** The result is `[3, 3, 5, 5, 6, 7]`. A naive scan of each window is O(n*k). In Spark SQL you'd write `MAX(x) OVER (ROWS BETWEEN k-1 PRECEDING AND CURRENT ROW)`.

---

## `min_subarray_len`
- **Problem:** Find the length of the shortest subarray of **positive** integers whose sum is at least `target`, or 0 if there isn't one.
- **Idea:** Use a variable-size window: grow on the right, and while the sum still meets the target, record the length and shrink from the left.
- **Complexity:** O(n) time, O(1) space

In [ ]:
def min_subarray_len(target: int, nums: List[int]) -> int:
    best = float("inf")
    left = total = 0
    for right, x in enumerate(nums):
        total += x
        while total >= target:
            best = min(best, right - left + 1)
            total -= nums[left]
            left += 1
    return 0 if best == float("inf") else int(best)


# Try it
print(min_subarray_len(7, [2, 3, 1, 2, 4, 3]))
print(min_subarray_len(100, [1, 2, 3]))

**What to notice:** The first answer is 2, from `[4, 3]`. This only works because values are positive, so shrinking always lowers the sum. With negatives, you need prefix sums (`subarray_sum_equals_k` in Part 5).

---

# Part 2: `dsa/strings.py`
String questions reuse the array patterns, plus counting and stacks.

In [ ]:
from collections import Counter
from typing import List

The imports are loaded.

---

## `is_palindrome`
- **Problem:** Check whether a string reads the same both ways, ignoring case and anything that isn't a letter or digit.
- **Idea:** Move two pointers inward, skipping characters that don't count.
- **Complexity:** O(n) time, O(1) space

In [ ]:
def is_palindrome(s: str) -> bool:
    i, j = 0, len(s) - 1
    while i < j:
        if not s[i].isalnum():
            i += 1
        elif not s[j].isalnum():
            j -= 1
        else:
            if s[i].lower() != s[j].lower():
                return False
            i, j = i + 1, j - 1
    return True


# Try it
print(is_palindrome("A man, a plan, a canal: Panama"))
print(is_palindrome("race a car"))

**What to notice:** Skipping in place avoids building a cleaned copy of the string, which is what keeps the space at O(1).

---

## `is_anagram`
- **Problem:** Check whether one string is a rearrangement of another.
- **Idea:** Anagrams have the same letter counts, so compare two `Counter`s. The length check exits early.
- **Complexity:** O(n) time, O(k) space for k distinct characters

In [ ]:
def is_anagram(a: str, b: str) -> bool:
    return len(a) == len(b) and Counter(a) == Counter(b)


# Try it
print(is_anagram("listen", "silent"))
print(is_anagram("aacc", "ccac"))

**What to notice:** `aacc` and `ccac` use the same letters with different counts, so comparing sets would wrongly say they match. Sorting also works, but it's O(n log n).

---

## `length_of_longest_unique_substring`
- **Problem:** Find the length of the longest substring with no repeated characters.
- **Idea:** Slide a window and remember each character's last index. On a repeat inside the window, jump `left` past the earlier copy.
- **Complexity:** O(n) time, O(min(n, alphabet)) space

In [ ]:
def length_of_longest_unique_substring(s: str) -> int:
    last_seen = {}
    left = best = 0
    for right, ch in enumerate(s):
        if ch in last_seen and last_seen[ch] >= left:
            left = last_seen[ch] + 1
        last_seen[ch] = right
        best = max(best, right - left + 1)
    return best


# Try it
print(length_of_longest_unique_substring("pwwkew"))
print(length_of_longest_unique_substring("abba"))

**What to notice:** `abba` returns 2. The `>= left` check matters here: the final `a`'s earlier copy is already outside the window, so `left` must not move back.

---

## `longest_palindromic_substring`
- **Problem:** Find the longest palindromic substring.
- **Idea:** Every palindrome grows out from a center. Try all 2n-1 centers: each character for odd lengths, and each gap for even lengths.
- **Complexity:** O(n^2) time, O(1) space

In [ ]:
def longest_palindromic_substring(s: str) -> str:
    if not s:
        return ""
    start, end = 0, 0

    def expand(l: int, r: int) -> tuple:
        while l >= 0 and r < len(s) and s[l] == s[r]:
            l, r = l - 1, r + 1
        return l + 1, r - 1

    for c in range(len(s)):
        for l, r in (expand(c, c), expand(c, c + 1)):
            if r - l > end - start:
                start, end = l, r
    return s[start:end + 1]


# Try it
print(longest_palindromic_substring("babad"))
print(longest_palindromic_substring("cbbd"))

**What to notice:** `cbbd` returns `bb`, which only the even-length center finds. Manacher's algorithm is O(n): mention it, but you're rarely asked to code it.

---

## `min_window_substring`
- **Problem:** Find the smallest substring of `s` that contains every character of `t`, including duplicates.
- **Idea:** `need` counts what's still owed, and `missing` counts how many characters are outstanding. Grow right until `missing` is 0, then shrink left past anything extra.
- **Complexity:** O(|s| + |t|) time

In [ ]:
def min_window_substring(s: str, t: str) -> str:
    if not t or not s:
        return ""
    need = Counter(t)
    missing = len(t)
    left = 0
    best = (0, float("inf"))
    for right, ch in enumerate(s):
        if need[ch] > 0:
            missing -= 1
        need[ch] -= 1
        if missing == 0:
            while need[s[left]] < 0:
                need[s[left]] += 1
                left += 1
            if right - left < best[1] - best[0]:
                best = (left, right)
            need[s[left]] += 1
            missing += 1
            left += 1
    return "" if best[1] == float("inf") else s[best[0]:best[1] + 1]


# Try it
print(min_window_substring("ADOBECODEBANC", "ABC"))
print(min_window_substring("a", "aa"))

**What to notice:** The answer is `BANC`. `need` can go negative, which marks a surplus character that the window can safely drop. This is a common hard interview question, so trace it by hand once.

---

## `compress`
- **Problem:** Run-length encode a string (`aaabcc` becomes `a3b1c2`), returning the original if encoding isn't shorter.
- **Idea:** Make one pass, tracking the current character and its run length. Collect the pieces in a list and join them at the end.
- **Complexity:** O(n) time

In [ ]:
def compress(s: str) -> str:
    if not s:
        return s
    parts: List[str] = []
    run_char, run_len = s[0], 1
    for ch in s[1:]:
        if ch == run_char:
            run_len += 1
        else:
            parts.append(f"{run_char}{run_len}")
            run_char, run_len = ch, 1
    parts.append(f"{run_char}{run_len}")
    out = "".join(parts)
    return out if len(out) < len(s) else s


# Try it
print(compress("aaabccccc"))
print(compress("abc"))

**What to notice:** `abc` stays as it is, because `a1b1c1` is longer. Parquet uses this same idea, run-length encoding, which is why clustered data compresses well.

---

## `reverse_words`
- **Problem:** Reverse the word order and normalize the spaces.
- **Idea:** `split()` with no argument handles repeated and leading spaces. Reverse the words, then join them with single spaces.
- **Complexity:** O(n) time

In [ ]:
def reverse_words(s: str) -> str:
    return " ".join(reversed(s.split()))


# Try it
print(reverse_words("  the sky  is blue "))

**What to notice:** A one-liner is fine here. If the interviewer asks for O(1) space on a mutable array, reverse the whole thing, then each word: the same trick as `rotate_right`.

---

## `valid_parentheses`
- **Problem:** Check that every bracket closes in the correct order.
- **Idea:** Use a **stack**. Push openers. Each closer must match the most recent opener.
- **Complexity:** O(n) time, O(n) space

In [ ]:
def valid_parentheses(s: str) -> bool:
    pairs = {")": "(", "]": "[", "}": "{"}
    stack: List[str] = []
    for ch in s:
        if ch in "([{":
            stack.append(ch)
        elif ch in pairs:
            if not stack or stack.pop() != pairs[ch]:
                return False
    return not stack


# Try it
print(valid_parentheses("([{}])"))
print(valid_parentheses("(]"))
print(valid_parentheses("("))

**What to notice:** The final `return not stack` catches `(`, which has no mismatch but never closes. Use a stack whenever the most recent item matters, as with nesting.

---

# Part 3: `dsa/binary_search.py`
Binary search applies to any problem where you can safely discard half the candidates, not just sorted lookups.

In [ ]:
from typing import List

The imports are loaded.

---

## `binary_search`
- **Problem:** Find the index of `target` in a sorted list, or return -1.
- **Idea:** Check the middle and discard the half that can't contain the target. The invariant: if the target exists, it's inside `nums[lo..hi]`.
- **Complexity:** O(log n) time, O(1) space

In [ ]:
def binary_search(nums: List[int], target: int) -> int:
    lo, hi = 0, len(nums) - 1
    while lo <= hi:
        mid = lo + (hi - lo) // 2  # overflow-safe habit from other languages
        if nums[mid] == target:
            return mid
        if nums[mid] < target:
            lo = mid + 1
        else:
            hi = mid - 1
    return -1


# Try it
print(binary_search([-1, 0, 3, 5, 9, 12], 9))
print(binary_search([-1, 0, 3, 5, 9, 12], 2))

**What to notice:** `lo <= hi` and `mid +/- 1` go together. With an inclusive `hi`, a one-element range still needs checking, and `mid` has already been ruled out.

---

## `lower_bound`
- **Problem:** Find the first index whose value is `>= target`. It's the same as `bisect.bisect_left`.
- **Idea:** Use a half-open range `[lo, hi)`. If `nums[mid]` is too small, the answer is to the right. Otherwise `mid` might be the answer, so keep it with `hi = mid`.
- **Complexity:** O(log n) time

In [ ]:
def lower_bound(nums: List[int], target: int) -> int:
    lo, hi = 0, len(nums)
    while lo < hi:
        mid = (lo + hi) // 2
        if nums[mid] < target:
            lo = mid + 1
        else:
            hi = mid
    return lo


# Try it
print(lower_bound([1, 3, 3, 5], 3))
print(lower_bound([1, 3, 3, 5], 4))
print(lower_bound([1, 3, 3, 5], 9))

**What to notice:** It finds the **first** 3 (index 1), and if the target isn't there it returns where it would go. Note the different template: `lo < hi` and `hi = mid`.

---

## `search_range`
- **Problem:** Find the first and last index of `target` in a sorted list.
- **Idea:** Run two boundary searches: the first position of `target`, and the first position of `target + 1`, minus one.
- **Complexity:** O(log n) time

In [ ]:
def search_range(nums: List[int], target: int) -> List[int]:
    first = lower_bound(nums, target)
    if first == len(nums) or nums[first] != target:
        return [-1, -1]
    last = lower_bound(nums, target + 1) - 1
    return [first, last]


# Try it
print(search_range([5, 7, 7, 8, 8, 10], 8))
print(search_range([5, 7, 7, 8, 8, 10], 6))

**What to notice:** This stays O(log n) however many duplicates there are. Scanning outward from a match would be O(n) in the worst case. It reuses `lower_bound` from the previous section.

---

## `search_rotated`
- **Problem:** Search a sorted list that has been rotated, with distinct values.
- **Idea:** At least one half is always sorted. Find which half, then check whether the target falls inside its range.
- **Complexity:** O(log n) time

In [ ]:
def search_rotated(nums: List[int], target: int) -> int:
    lo, hi = 0, len(nums) - 1
    while lo <= hi:
        mid = (lo + hi) // 2
        if nums[mid] == target:
            return mid
        if nums[lo] <= nums[mid]:  # left half sorted
            if nums[lo] <= target < nums[mid]:
                hi = mid - 1
            else:
                lo = mid + 1
        else:  # right half sorted
            if nums[mid] < target <= nums[hi]:
                lo = mid + 1
            else:
                hi = mid - 1
    return -1


# Try it
print(search_rotated([4, 5, 6, 7, 0, 1, 2], 0))
print(search_rotated([4, 5, 6, 7, 0, 1, 2], 3))

**What to notice:** The first answer is index 4. Comparing `nums[lo] <= nums[mid]` tells you the left half is sorted. This is one of the most common binary search interview questions.

---

## `find_min_rotated`
- **Problem:** Find the minimum of a rotated sorted list with distinct values.
- **Idea:** Compare the middle with the **right** end. If `nums[mid] > nums[hi]`, the drop is to the right of `mid`. Otherwise the minimum is at `mid` or to its left.
- **Complexity:** O(log n) time

In [ ]:
def find_min_rotated(nums: List[int]) -> int:
    lo, hi = 0, len(nums) - 1
    while lo < hi:
        mid = (lo + hi) // 2
        if nums[mid] > nums[hi]:
            lo = mid + 1
        else:
            hi = mid
    return nums[lo]


# Try it
print(find_min_rotated([3, 4, 5, 1, 2]))
print(find_min_rotated([11, 13, 15, 17]))

**What to notice:** It also handles a list that was never rotated. The loop ends with `lo == hi` on the minimum, using the same `lo < hi` template as `lower_bound`.

---

## `int_sqrt`
- **Problem:** Compute `floor(sqrt(x))` without `math.sqrt`.
- **Idea:** Binary search over possible answers, using `mid * mid <= x` as the test.
- **Complexity:** O(log x) time

In [ ]:
def int_sqrt(x: int) -> int:
    lo, hi = 0, x
    while lo <= hi:
        mid = (lo + hi) // 2
        if mid * mid <= x:
            lo = mid + 1
        else:
            hi = mid - 1
    return hi


# Try it
print(int_sqrt(10))
print(int_sqrt(16))
print(int_sqrt(0))

**What to notice:** When the loop ends, `hi` is the largest value that passed, which is why the function returns `hi` rather than `lo`.

---

## `search_matrix`
- **Problem:** Search a matrix where rows are sorted and each row starts after the previous row ends.
- **Idea:** Treat it as one flat sorted list. Index `mid` maps to row `mid // n` and column `mid % n`.
- **Complexity:** O(log(m*n)) time

In [ ]:
def search_matrix(matrix: List[List[int]], target: int) -> bool:
    if not matrix or not matrix[0]:
        return False
    m, n = len(matrix), len(matrix[0])
    lo, hi = 0, m * n - 1
    while lo <= hi:
        mid = (lo + hi) // 2
        val = matrix[mid // n][mid % n]
        if val == target:
            return True
        if val < target:
            lo = mid + 1
        else:
            hi = mid - 1
    return False


# Try it
matrix = [[1, 3, 5, 7], [10, 11, 16, 20], [23, 30, 34, 60]]
print(search_matrix(matrix, 3))
print(search_matrix(matrix, 13))

**What to notice:** No special 2D logic is needed once you see the mapping. That index conversion is the whole trick.

---

## `ship_within_days`
- **Problem:** Find the smallest ship capacity that delivers packages, in order, within `days`.
- **Idea:** This is **binary search on the answer**. If a capacity works, every larger one works too, so search for the first that passes. The range runs from the heaviest package to the total weight.
- **Complexity:** O(n log(sum(weights))) time

In [ ]:
def ship_within_days(weights: List[int], days: int) -> int:
    def feasible(cap: int) -> bool:
        needed, load = 1, 0
        for w in weights:
            if load + w > cap:
                needed += 1
                load = 0
            load += w
        return needed <= days

    lo, hi = max(weights), sum(weights)
    while lo < hi:
        mid = (lo + hi) // 2
        if feasible(mid):
            hi = mid
        else:
            lo = mid + 1
    return lo


# Try it
print(ship_within_days([1, 2, 3, 4, 5, 6, 7, 8, 9, 10], 5))
print(ship_within_days([3, 2, 2, 4, 1, 4], 3))

**What to notice:** The first answer is 15. Look for the phrase "minimum X such that...". "Smallest cluster that meets the SLA" has the same shape.

---

## `find_peak_element`
- **Problem:** Find the index of any element larger than both neighbors.
- **Idea:** If `nums[mid] < nums[mid + 1]`, the values rise to the right, so a peak must exist there. Otherwise a peak is at `mid` or to its left.
- **Complexity:** O(log n) time

In [ ]:
def find_peak_element(nums: List[int]) -> int:
    lo, hi = 0, len(nums) - 1
    while lo < hi:
        mid = (lo + hi) // 2
        if nums[mid] < nums[mid + 1]:
            lo = mid + 1
        else:
            hi = mid
    return lo


# Try it
print(find_peak_element([1, 2, 1, 3, 5, 6, 4]))

**What to notice:** It returns index 5 (value 6), even though index 1 is also a peak: any peak is acceptable. This shows binary search doesn't need fully sorted data, only a way to discard a half.

---

# Part 4: `dsa/graphs.py`
Graphs are stored as adjacency lists: a dict from each node to its neighbors. Job dependencies, lineage and entity resolution are all graph problems.

In [ ]:
import heapq
from collections import defaultdict, deque
from typing import Dict, Hashable, List, Optional, Tuple
Graph = Dict[Hashable, List[Hashable]]

The imports are loaded.

---

## `build_graph`
- **Problem:** Turn an edge list into an adjacency list.
- **Idea:** Add each edge once if it's directed, or in both directions if not. For directed graphs, `setdefault` makes sure nodes with no outgoing edges still appear.
- **Complexity:** O(V + E) time and space

In [ ]:
def build_graph(edges: List[Tuple[Hashable, Hashable]], directed: bool = False) -> Graph:
    g: Graph = defaultdict(list)
    for u, v in edges:
        g[u].append(v)
        if not directed:
            g[v].append(u)
        else:
            g.setdefault(v, [])
    return dict(g)


# Try it
edges = [("A", "B"), ("A", "C"), ("B", "D"), ("C", "D"), ("D", "E")]
graph = build_graph(edges)
print(graph)
print(build_graph([(1, 2)], directed=True))

**What to notice:** In the directed example, node 2 appears with an empty list. Without that, code that looks up `graph[2]` would fail. `graph` is used in the next few sections.

---

## `bfs_order`
- **Problem:** List nodes in breadth-first order.
- **Idea:** Use a queue and a visited set, and mark nodes visited **when you add them**, so none is queued twice.
- **Complexity:** O(V + E) time, O(V) space

In [ ]:
def bfs_order(graph: Graph, start: Hashable) -> List[Hashable]:
    visited = {start}
    order = []
    q = deque([start])
    while q:
        node = q.popleft()
        order.append(node)
        for nb in graph.get(node, []):
            if nb not in visited:
                visited.add(nb)
                q.append(nb)
    return order


# Try it
print(bfs_order(graph, "A"))

**What to notice:** BFS visits in rings: A, then its neighbors B and C, then D, then E.

---

## `dfs_order`
- **Problem:** List nodes in depth-first order, without recursion.
- **Idea:** Use an explicit stack, and push neighbors in reverse so they're visited in list order.
- **Complexity:** O(V + E) time

In [ ]:
def dfs_order(graph: Graph, start: Hashable) -> List[Hashable]:
    visited = set()
    order = []
    stack = [start]
    while stack:
        node = stack.pop()
        if node in visited:
            continue
        visited.add(node)
        order.append(node)
        for nb in reversed(graph.get(node, [])):
            if nb not in visited:
                stack.append(nb)
    return order


# Try it
print(dfs_order(graph, "A"))

**What to notice:** DFS goes deep first: A, B, D, then C and E. The explicit stack avoids Python's recursion limit of about 1,000 on deep graphs, and here the visited check happens when a node is popped.

---

## `shortest_path`
- **Problem:** Find a path with the fewest edges in an unweighted graph.
- **Idea:** BFS reaches each node first by a shortest path. Record each node's parent, then walk back from the goal.
- **Complexity:** O(V + E) time

In [ ]:
def shortest_path(graph: Graph, start: Hashable, goal: Hashable) -> Optional[List[Hashable]]:
    parent = {start: None}
    q = deque([start])
    while q:
        node = q.popleft()
        if node == goal:
            path = []
            while node is not None:
                path.append(node)
                node = parent[node]
            return path[::-1]
        for nb in graph.get(node, []):
            if nb not in parent:
                parent[nb] = node
                q.append(nb)
    return None


# Try it
print(shortest_path(graph, "A", "E"))
print(shortest_path(graph, "A", "Z"))

**What to notice:** The result is built backward from the goal, then reversed with `[::-1]`. `parent` also acts as the visited set. Use Dijkstra if edges have different costs.

---

## `num_islands`
- **Problem:** Count connected groups of `"1"` in a grid.
- **Idea:** A grid is a graph where each cell's neighbors are its 4 adjacent cells. Each unvisited land cell starts a new island, and a flood fill marks the rest of it.
- **Complexity:** O(rows * cols) time

In [ ]:
def num_islands(grid: List[List[str]]) -> int:
    if not grid:
        return 0
    rows, cols = len(grid), len(grid[0])
    seen = set()
    count = 0
    for r in range(rows):
        for c in range(cols):
            if grid[r][c] == "1" and (r, c) not in seen:
                count += 1
                seen.add((r, c))
                q = deque([(r, c)])
                while q:
                    cr, cc = q.popleft()
                    for dr, dc in ((1, 0), (-1, 0), (0, 1), (0, -1)):
                        nr, nc = cr + dr, cc + dc
                        if 0 <= nr < rows and 0 <= nc < cols and grid[nr][nc] == "1" and (nr, nc) not in seen:
                            seen.add((nr, nc))
                            q.append((nr, nc))
    return count


# Try it
grid = [list("11000"), list("11000"), list("00100"), list("00011")]
print(num_islands(grid))

**What to notice:** There are 3 islands. The `0 <= nr < rows` bounds check comes before reading `grid[nr][nc]`, which avoids index errors and wraparound from negative indexes.

---

## `has_cycle_directed`
- **Problem:** Detect a cycle in a directed graph.
- **Idea:** Use DFS with three colors: WHITE is unvisited, GRAY is on the current path, BLACK is finished. An edge to a GRAY node goes back into the current path, so it's a cycle.
- **Complexity:** O(V + E) time

In [ ]:
def has_cycle_directed(graph: Graph) -> bool:
    WHITE, GRAY, BLACK = 0, 1, 2
    color = {n: WHITE for n in graph}
    for root in graph:
        if color[root] != WHITE:
            continue
        stack = [(root, iter(graph.get(root, [])))]
        color[root] = GRAY
        while stack:
            node, it = stack[-1]
            nxt = next(it, None)
            if nxt is None:
                color[node] = BLACK
                stack.pop()
            elif color.get(nxt, WHITE) == GRAY:
                return True
            elif color.get(nxt, WHITE) == WHITE:
                color[nxt] = GRAY
                stack.append((nxt, iter(graph.get(nxt, []))))
    return False


# Try it
print(has_cycle_directed({1: [2], 2: [3], 3: [1]}))
print(has_cycle_directed({1: [2, 3], 2: [3], 3: []}))

**What to notice:** In the second graph, node 3 is reached twice without a cycle. A plain visited set would wrongly flag this, which is why GRAY and BLACK are separate.

---

## `topological_sort`
- **Problem:** Order tasks so every dependency comes first, or return `None` if there's a cycle.
- **Idea:** This is **Kahn's algorithm**: repeatedly take a node with in-degree 0 and reduce its children's in-degrees. A heap makes the tie-breaking deterministic.
- **Complexity:** O(V log V + E) with the heap, O(V + E) with a plain queue

In [ ]:
def topological_sort(graph: Graph) -> Optional[List[Hashable]]:
    indeg = {n: 0 for n in graph}
    for u in graph:
        for v in graph[u]:
            indeg[v] = indeg.get(v, 0) + 1
    ready = [n for n, d in indeg.items() if d == 0]
    heapq.heapify(ready)
    order = []
    while ready:
        node = heapq.heappop(ready)
        order.append(node)
        for v in graph.get(node, []):
            indeg[v] -= 1
            if indeg[v] == 0:
                heapq.heappush(ready, v)
    return order if len(order) == len(indeg) else None


# Try it
dag = {"raw": ["stg"], "stg": ["dim", "fact"], "dim": ["fact"], "fact": ["gold"], "gold": []}
print(topological_sort(dag))
print(topological_sort({"a": ["b"], "b": ["a"]}))

**What to notice:** `fact` waits for both `stg` and `dim`. If some nodes never reach in-degree 0, there's a cycle, so the code returns `None`. Job schedulers order task DAGs the same way.

---

## `dijkstra`
- **Problem:** Find the cheapest distance from `start` to every reachable node, with non-negative weights.
- **Idea:** Use a min-heap of `(distance, node)`. Instead of updating entries, push new ones and skip stale ones when they pop.
- **Complexity:** O((V + E) log V) time

In [ ]:
def dijkstra(weighted: Dict[Hashable, List[Tuple[Hashable, float]]], start: Hashable) -> Dict[Hashable, float]:
    dist = {start: 0}
    heap = [(0, start)]
    while heap:
        d, node = heapq.heappop(heap)
        if d > dist.get(node, float("inf")):
            continue
        for nb, w in weighted.get(node, []):
            nd = d + w
            if nd < dist.get(nb, float("inf")):
                dist[nb] = nd
                heapq.heappush(heap, (nd, nb))
    return dist


# Try it
weighted = {"A": [("B", 4), ("C", 1)], "C": [("B", 2), ("D", 5)], "B": [("D", 1)], "D": []}
print(dijkstra(weighted, "A"))

**What to notice:** B costs 3 through C, not 4 directly, and D costs 4 through C and B. Negative weights break it; use Bellman-Ford for those.

---

## `UnionFind`
- **Problem:** Track groups that merge over time, and check whether two items are in the same group.
- **Idea:** Each item points to a parent, and the root names the group. `find` compresses paths as it walks, and `union` attaches the smaller group to the larger.
- **Complexity:** Nearly O(1) per operation (amortized)

In [ ]:
class UnionFind:

    def __init__(self) -> None:
        self.parent: Dict[Hashable, Hashable] = {}
        self.size: Dict[Hashable, int] = {}

    def find(self, x: Hashable) -> Hashable:
        if x not in self.parent:
            self.parent[x] = x
            self.size[x] = 1
        root = x
        while self.parent[root] != root:
            root = self.parent[root]
        while self.parent[x] != root:  # path compression
            self.parent[x], x = root, self.parent[x]
        return root

    def union(self, a: Hashable, b: Hashable) -> bool:
        ra, rb = self.find(a), self.find(b)
        if ra == rb:
            return False
        if self.size[ra] < self.size[rb]:
            ra, rb = rb, ra
        self.parent[rb] = ra
        self.size[ra] += self.size[rb]
        return True


# Try it
uf = UnionFind()
uf.union("email:a@x.com", "phone:555")
uf.union("phone:555", "crm:42")
print(uf.find("email:a@x.com") == uf.find("crm:42"))
print(uf.find("email:a@x.com") == uf.find("crm:99"))

**What to notice:** Email and CRM records link through the shared phone number even though they never met directly. This is entity resolution: building one customer from several source systems.

---

## `count_components`
- **Problem:** Count connected groups among nodes `0..n-1`.
- **Idea:** Start with `n` groups. Each `union` that joins two different groups reduces the count by one.
- **Complexity:** Nearly O(n + edges)

In [ ]:
def count_components(n: int, edges: List[Tuple[int, int]]) -> int:
    uf = UnionFind()
    for i in range(n):
        uf.find(i)
    merges = sum(1 for a, b in edges if uf.union(a, b))
    return n - merges


# Try it
print(count_components(5, [(0, 1), (1, 2), (3, 4)]))
print(count_components(3, []))

**What to notice:** `union` returns `False` when both items are already together, so counting the `True` results gives the number of real merges. It reuses `UnionFind` from the previous section.

---

# Part 5: `dsa/hash_maps.py`
Hash maps trade memory for O(1) average lookups. They also underlie Spark's hash joins, hash aggregation and shuffles.

In [ ]:
from collections import Counter, OrderedDict, defaultdict
from typing import Dict, Hashable, List

The imports are loaded.

---

## `two_sum`
- **Problem:** Find indices of two numbers that add up to `target`, in an **unsorted** list.
- **Idea:** For each value, check whether its complement has already been seen, then store the value and its index.
- **Complexity:** O(n) time, O(n) space (brute force is O(n^2))

In [ ]:
def two_sum(nums: List[int], target: int) -> List[int]:
    seen: Dict[int, int] = {}
    for j, x in enumerate(nums):
        if target - x in seen:
            return [seen[target - x], j]
        seen[x] = j
    return []


# Try it
print(two_sum([2, 7, 11, 15], 9))
print(two_sum([3, 3], 6))

**What to notice:** Storing **after** checking means a value can't pair with itself, yet `[3, 3]` still works because the second 3 finds the first.

---

## `group_anagrams`
- **Problem:** Group words that are anagrams of each other.
- **Idea:** Give every anagram the same key (its sorted letters), and put each word in that key's bucket.
- **Complexity:** O(n * k log k) time for n words of length k

In [ ]:
def group_anagrams(words: List[str]) -> List[List[str]]:
    groups: Dict[str, List[str]] = defaultdict(list)
    for w in words:
        groups["".join(sorted(w))].append(w)
    return list(groups.values())


# Try it
print(group_anagrams(["eat", "tea", "tan", "ate", "nat", "bat"]))

**What to notice:** Choosing a canonical key is the whole idea, and it's exactly what `groupBy` does with a grouping expression. A 26-letter count tuple as the key would make each word O(k).

---

## `top_k_frequent`
- **Problem:** Return the `k` most frequent values, breaking ties with the smaller value.
- **Idea:** Count the values, then put them into buckets indexed by frequency. Read the buckets from highest to lowest.
- **Complexity:** O(n) time, plus sorting within each bucket for the ties

In [ ]:
def top_k_frequent(nums: List[int], k: int) -> List[int]:
    counts = Counter(nums)
    buckets: List[List[int]] = [[] for _ in range(len(nums) + 1)]
    for val, c in counts.items():
        buckets[c].append(val)
    out: List[int] = []
    for c in range(len(buckets) - 1, 0, -1):
        for val in sorted(buckets[c]):
            out.append(val)
            if len(out) == k:
                return out
    return out


# Try it
print(top_k_frequent([1, 1, 1, 2, 2, 3], 2))
print(top_k_frequent([4, 4, 5, 5, 6], 2))

**What to notice:** A value can't appear more than n times, so there are only n+1 buckets. That bound is what avoids a full sort. `heapq.nlargest` is the simpler O(n log k) alternative.

---

## `longest_consecutive`
- **Problem:** Find the length of the longest run of consecutive integers in an unsorted list.
- **Idea:** Put everything in a set. Only start counting at a run's beginning, where `x - 1` isn't in the set.
- **Complexity:** O(n) time

In [ ]:
def longest_consecutive(nums: List[int]) -> int:
    s = set(nums)
    best = 0
    for x in s:
        if x - 1 not in s:
            length = 1
            while x + length in s:
                length += 1
            best = max(best, length)
    return best


# Try it
print(longest_consecutive([100, 4, 200, 1, 3, 2]))
print(longest_consecutive([]))

**What to notice:** It looks like a nested loop, but each run is walked only once, from its start, so the total is O(n). In SQL the same problem is solved with gaps and islands.

---

## `subarray_sum_equals_k`
- **Problem:** Count contiguous subarrays summing to `k`, with negatives allowed.
- **Idea:** A subarray's sum is the difference of two prefix sums. For each prefix, count how many earlier prefixes equal `prefix - k`.
- **Complexity:** O(n) time, O(n) space

In [ ]:
def subarray_sum_equals_k(nums: List[int], k: int) -> int:
    counts = Counter({0: 1})
    prefix = total = 0
    for x in nums:
        prefix += x
        total += counts[prefix - k]
        counts[prefix] += 1
    return total


# Try it
print(subarray_sum_equals_k([1, 1, 1], 2))
print(subarray_sum_equals_k([1, -1, 0], 0))

**What to notice:** Starting the counter with `{0: 1}` covers subarrays that begin at index 0. A sliding window can't do this, because negatives mean growing the window doesn't always raise the sum.

---

## `first_unique_char`
- **Problem:** Find the index of the first character that appears exactly once, or return -1.
- **Idea:** Make two passes: count every character, then find the first with a count of 1.
- **Complexity:** O(n) time

In [ ]:
def first_unique_char(s: str) -> int:
    counts = Counter(s)
    for i, ch in enumerate(s):
        if counts[ch] == 1:
            return i
    return -1


# Try it
print(first_unique_char("loveleetcode"))
print(first_unique_char("aabb"))

**What to notice:** The second pass goes over the **string**, not the counter, because you need the first position in the original order.

---

## `LRUCache`
- **Problem:** Build a cache that evicts the least recently used key, with O(1) `get` and `put`.
- **Idea:** An `OrderedDict` combines a hash map with a linked list. Move a key to the end on every use, and evict from the front.
- **Complexity:** O(1) per operation

In [ ]:
class LRUCache:

    def __init__(self, capacity: int) -> None:
        self.capacity = capacity
        self.data: "OrderedDict[Hashable, object]" = OrderedDict()

    def get(self, key: Hashable) -> object:
        if key not in self.data:
            return -1
        self.data.move_to_end(key)
        return self.data[key]

    def put(self, key: Hashable, value: object) -> None:
        if key in self.data:
            self.data.move_to_end(key)
        self.data[key] = value
        if len(self.data) > self.capacity:
            self.data.popitem(last=False)


# Try it
cache = LRUCache(2)
cache.put(1, "a")
cache.put(2, "b")
print(cache.get(1))
cache.put(3, "c")
print(cache.get(2))
print(list(cache.data))

**What to notice:** `get(1)` made key 1 recent, so adding key 3 evicted key 2. Moving the key on `put` matters too: without it, an updated key keeps its old position. `30_senior_walkthrough` builds this from scratch.

---

## `hash_partition`
- **Problem:** Assign keys to partitions using `hash(key) % n`, which is what a Spark shuffle does.
- **Idea:** The same key always lands in the same partition. That's what makes joins and aggregations correct after a shuffle.
- **Complexity:** O(n) time

In [ ]:
def hash_partition(keys: List[Hashable], num_partitions: int) -> Dict[int, List[Hashable]]:
    parts: Dict[int, List[Hashable]] = defaultdict(list)
    for k in keys:
        parts[hash(k) % num_partitions].append(k)
    return dict(parts)


# Try it
parts = hash_partition(list(range(100)) + ["hot"] * 50, 8)
sizes = sorted((len(v) for v in parts.values()), reverse=True)
print("rows per partition:", sizes)

**What to notice:** All 50 `hot` rows land in one partition, which is **data skew**. String hashes in Python change between runs, so which partition gets them varies, and so do the sizes. Spark uses Murmur3, which is deterministic.

---

# Part 6: `dsa/python_optimizations.py`
Each pair returns the same result, so you can compare only the cost. `compare_pair` below is a helper for this section: it checks that the results match, then times both.

The file also holds `PAIRS`, a dict of these pairs that `benchmark.compare` uses in Part 7.

In [ ]:
import time
import bisect
import heapq
from collections import Counter
from functools import lru_cache
from typing import Callable, Dict, List, Tuple

def compare_pair(slow, fast, *args):
    t0 = time.perf_counter()
    slow_result = slow(*args)
    t_slow = time.perf_counter() - t0
    t0 = time.perf_counter()
    fast_result = fast(*args)
    t_fast = time.perf_counter() - t0
    assert slow_result == fast_result
    print(f"slow: {t_slow * 1000:9.3f} ms\nfast: {t_fast * 1000:9.3f} ms\nspeedup: {t_slow / max(t_fast, 1e-9):,.1f}x")

The imports are loaded.

---

## `common_items_slow` and `common_items_fast`
- **Problem:** Find the items of `a` that are also in `b`.
- **Idea:** `x in list` scans the list, but `x in set` is a hash lookup. Build the set once, outside the loop.
- **Complexity:** O(a * b) vs O(a + b)

In [ ]:
def common_items_slow(a: List[int], b: List[int]) -> List[int]:
    return [x for x in a if x in b]          # O(len(a) * len(b))


def common_items_fast(a: List[int], b: List[int]) -> List[int]:
    bs = set(b)                               # O(len(b)) once
    return [x for x in a if x in bs]          # O(len(a))


# Try it
a = list(range(3000))
b = list(range(1500, 4500))
compare_pair(common_items_slow, common_items_fast, a, b)

**What to notice:** This is the most common Python performance bug: `in` on a list inside a loop. It's also the reason hash joins beat nested-loop joins.

---

## `build_csv_slow` and `build_csv_fast`
- **Problem:** Build a large string from many rows.
- **Idea:** `+=` can copy the whole string on each step, while `"".join` allocates once.
- **Complexity:** Up to O(n^2) vs O(n)

In [ ]:
def build_csv_slow(rows: List[Tuple]) -> str:
    out = ""
    for r in rows:
        out += ",".join(map(str, r)) + "\n"
    return out


def build_csv_fast(rows: List[Tuple]) -> str:
    return "".join(",".join(map(str, r)) + "\n" for r in rows)


# Try it
rows = [(i, i * 2, "x") for i in range(3000)]
compare_pair(build_csv_slow, build_csv_fast, rows)

**What to notice:** Expect a **small** gap. CPython can often grow a string in place when nothing else references it, but that's an implementation detail, so `join` is the portable answer. Mentioning this is good interview nuance.

---

## `count_words_slow` and `count_words_fast`
- **Problem:** Count how often each word appears.
- **Idea:** `Counter` runs its counting loop in C, while the slow version does extra lookups in Python.
- **Complexity:** O(n) for both, with a smaller constant for `Counter`

In [ ]:
def count_words_slow(words: List[str]) -> Dict[str, int]:
    counts: Dict[str, int] = {}
    for w in words:
        if w in counts:
            counts[w] = counts[w] + 1
        else:
            counts[w] = 1
    return counts


def count_words_fast(words: List[str]) -> Dict[str, int]:
    return dict(Counter(words))


# Try it
words = [f"w{i % 100}" for i in range(50000)]
compare_pair(count_words_slow, count_words_fast, words)

**What to notice:** Both are O(n), so this is a constant-factor win. Big-O isn't the whole story: built-ins are usually faster and shorter.

---

## `top_k_slow` and `top_k_fast`
- **Problem:** Get the `k` largest values.
- **Idea:** Sorting orders everything, even though you only need `k` values. A heap tracks just those.
- **Complexity:** O(n log n) vs O(n log k)

In [ ]:
def top_k_slow(nums: List[int], k: int) -> List[int]:
    return sorted(nums, reverse=True)[:k]


def top_k_fast(nums: List[int], k: int) -> List[int]:
    return heapq.nlargest(k, nums)


# Try it
import random

rng = random.Random(1)
shuffled = [rng.randint(0, 10**9) for _ in range(100000)]
print("Shuffled data:")
compare_pair(top_k_slow, top_k_fast, shuffled, 10)
print("Data already in ascending order:")
compare_pair(top_k_slow, top_k_fast, list(range(100000)), 10)

**What to notice:** On shuffled data the heap wins, and its lead grows with n. On ascending data the result **flips**: Python's sort takes advantage of existing order and is close to O(n), while the heap has to replace its contents on every element. Big-O describes growth, so real data can still change the winner. Spark's `orderBy().limit(k)` also keeps a top-k per partition.

---

## `rank_lookups_slow` and `rank_lookups_fast`
- **Problem:** For each query, count how many sorted values are smaller.
- **Idea:** Sorted data lets `bisect` answer each query with a binary search instead of a scan.
- **Complexity:** O(q * n) vs O(q log n)

In [ ]:
def rank_lookups_slow(sorted_vals: List[int], queries: List[int]) -> List[int]:
    return [sum(1 for v in sorted_vals if v < q) for q in queries]


def rank_lookups_fast(sorted_vals: List[int], queries: List[int]) -> List[int]:
    return [bisect.bisect_left(sorted_vals, q) for q in queries]


# Try it
sorted_vals = list(range(0, 20000, 2))
queries = list(range(0, 20000, 100))
compare_pair(rank_lookups_slow, rank_lookups_fast, sorted_vals, queries)

**What to notice:** Paying once to sort makes every later lookup cheap. Delta's file statistics apply a similar idea: a range check rules out whole files.

---

## `fib_slow` and `fib_fast`
- **Problem:** Compute Fibonacci numbers.
- **Idea:** Naive recursion recomputes the same values repeatedly. `lru_cache` stores each result, so each value is computed once.
- **Complexity:** O(2^n) vs O(n)

In [ ]:
def fib_slow(n: int) -> int:
    return n if n < 2 else fib_slow(n - 1) + fib_slow(n - 2)   # O(2^n)


@lru_cache(maxsize=None)
def fib_fast(n: int) -> int:
    return n if n < 2 else fib_fast(n - 1) + fib_fast(n - 2)   # O(n)


# Try it
fib_fast.cache_clear()
compare_pair(fib_slow, fib_fast, 25)

**What to notice:** This is **memoization**. The warehouse version of the same trade-off is a materialized view: store a result once rather than recomputing it.

---

## `join_slow` and `join_fast`
- **Problem:** Join orders `(order_id, cust_id)` to customers `(cust_id, name)`.
- **Idea:** A nested loop compares every pair. A hash join builds a dict from the small side, then probes it once per order.
- **Complexity:** O(n * m) vs O(n + m)

In [ ]:
def join_slow(orders: List[Tuple[int, int]], customers: List[Tuple[int, str]]) -> List[Tuple[int, str]]:
    return [(o, name) for o, c in orders for cid, name in customers if c == cid]  # O(n*m)


def join_fast(orders: List[Tuple[int, int]], customers: List[Tuple[int, str]]) -> List[Tuple[int, str]]:
    lookup = dict(customers)                         # build side (the "broadcast" table)
    return [(o, lookup[c]) for o, c in orders if c in lookup]   # probe side: O(n + m)


# Try it
orders = [(i, i % 500) for i in range(3000)]
customers = [(c, f"c{c}") for c in range(500)]
compare_pair(join_slow, join_fast, orders, customers)

**What to notice:** This is a **broadcast hash join** in miniature: the dict is the broadcast table. It's a strong answer to "how does a join work?"

---

# Part 7: `dsa/benchmark.py`
These tools measure complexity from real timings instead of just claiming it. `02_python_complexity_lab` uses them on bigger experiments.

In [ ]:
import math
import random
import time
from typing import Callable, Dict, List, Sequence, Tuple
from dsa import python_optimizations as po

The imports are loaded.

---

## `time_call`
- **Problem:** Time a function call reliably.
- **Idea:** Run it several times and keep the **best** time. Noise from garbage collection or other processes only ever makes a run slower.
- **Complexity:** O(repeat) runs of the function

In [ ]:
def time_call(fn: Callable, *args, repeat: int = 3) -> float:
    best = float("inf")
    for _ in range(repeat):
        t0 = time.perf_counter()
        fn(*args)
        best = min(best, time.perf_counter() - t0)
    return best


# Try it
seconds = time_call(sum, range(100000))
print(f"{seconds:.6f}s")

**What to notice:** Taking the minimum rather than the average gives the most stable estimate of the code's actual cost.

---

## `growth`
- **Problem:** Measure how the runtime changes as the input grows.
- **Idea:** Call `time_call` at each size and return `(n, seconds)` pairs.
- **Complexity:** Depends on the function being measured

In [ ]:
def growth(fn_of_n: Callable[[int], object], sizes: Sequence[int], repeat: int = 3) -> List[Tuple[int, float]]:
    return [(n, time_call(fn_of_n, n, repeat=repeat)) for n in sizes]


# Try it
rows = growth(lambda n: sum(range(n)), [10000, 100000, 1000000])
for n, t in rows:
    print(f"n={n:>8,}  {t:.6f}s")

**What to notice:** Each input here is 10x bigger, so a linear function should take about 10x longer. The exact numbers change from run to run.

---

## `estimate_exponent`
- **Problem:** Estimate the `k` in O(n^k) from timing data.
- **Idea:** If `time = c * n^k`, then `log(time) = log(c) + k * log(n)`, a straight line whose slope is `k`. Least squares finds the slope.
- **Complexity:** O(number of rows)

In [ ]:
def estimate_exponent(rows: List[Tuple[int, float]]) -> float:
    pts = [(math.log(n), math.log(max(t, 1e-9))) for n, t in rows]
    mx = sum(x for x, _ in pts) / len(pts)
    my = sum(y for _, y in pts) / len(pts)
    num = sum((x - mx) * (y - my) for x, y in pts)
    den = sum((x - mx) ** 2 for x, _ in pts)
    return num / den if den else 0.0


# Try it
quadratic = [(1000, 1.0), (2000, 4.0), (4000, 16.0)]
print(round(estimate_exponent(quadratic), 2))
print(round(estimate_exponent(growth(lambda n: sum(range(n)), [100000, 200000, 400000, 800000])), 2))

**What to notice:** The made-up quadratic data gives exactly 2.0, and the real `sum` gives roughly 1. As a guide, about 0 is constant, 1 is linear and 2 is quadratic. O(n log n) shows up slightly above 1.

---

## `_inputs`
- **Problem:** Build the sample inputs for every slow vs fast pair.
- **Idea:** It returns a dict from each pair's name to its arguments. A fixed random seed makes the data the same on every run.
- **Complexity:** O(n) to build the data

In [ ]:
def _inputs(n: int) -> Dict[str, tuple]:
    rng = random.Random(42)
    nums = [rng.randint(0, n) for _ in range(n)]
    words = [f"w{rng.randint(0, n // 10 + 1)}" for _ in range(n)]
    return {
        "membership (list vs set)": (nums, nums[: n // 2]),
        "string build (+= vs join)": ([(i, i * 2, "x") for i in range(n)],),
        "counting (dict vs Counter)": (words,),
        "top-k (sort vs heap)": (nums, 10),
        "rank lookups (scan vs bisect)": (sorted(nums), nums[:200]),
        "fibonacci (naive vs memo)": (min(24, 10 + n // 500),),
        "join (nested loop vs hash)": ([(i, i % 500) for i in range(n)], [(c, f"c{c}") for c in range(500)]),
    }


# Try it
for name, args in _inputs(100).items():
    print(f"{name:<32} {len(args)} argument(s)")

**What to notice:** The leading underscore marks it as an internal helper, since only `compare` calls it.

---

## `compare`
- **Problem:** Run all 7 pairs from Part 6 at one input size and report the speedup.
- **Idea:** For each pair, confirm that both versions return the same result, then time each one.
- **Complexity:** Dominated by the slowest pair

In [ ]:
def compare(n: int = 2_000) -> List[Dict[str, object]]:
    rows = []
    inputs = _inputs(n)
    for name, (slow, fast) in po.PAIRS.items():
        args = inputs[name]
        po.fib_fast.cache_clear()
        assert slow(*args) == fast(*args), name
        po.fib_fast.cache_clear()
        t_slow = time_call(slow, *args, repeat=1)
        po.fib_fast.cache_clear()
        t_fast = time_call(fast, *args, repeat=1)
        rows.append({
            "technique": name,
            "n": n,
            "slow_ms": round(t_slow * 1000, 3),
            "fast_ms": round(t_fast * 1000, 3),
            "speedup_x": round(t_slow / max(t_fast, 1e-9), 1),
        })
    return rows


# Try it
for row in compare(n=1000):
    print(row)

**What to notice:** The `assert` matters: a speedup means nothing if the fast version returns a different answer. `cache_clear()` resets `fib_fast` so its cached results don't make it look faster than it is.

---

# Summary
You've now read every function in `dsa/`. To lock it in:
1. Pick a topic in `01_dsa_practice` and solve its problems in `exercises/` without looking back here.
2. Use `docs/cheatsheet.md` to practice naming the pattern from the problem statement alone.
3. For each function, say the idea and complexity out loud in under 30 seconds.